# Forecasting strategies for power demand 🔮⚡️
*Aligned to practicum10_time_series_III structure; uses the same data path.*

**Data file expected:** `data/electricity-production and consumption_2022.csv`

Notes:
- Keep the relative path as in your practicum (no changes needed).
- If you run in Colab, upload or mount the repo so the path exists.


In [ ]:
%%capture
!pip install -q skforecast==0.13.0 xgboost statsmodels

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import timedelta
from statsmodels.graphics.tsaplots import plot_acf
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from skforecast.ForecasterAutoreg import ForecasterAutoreg
from skforecast.ForecasterAutoregDirect import ForecasterAutoregDirect
plt.rcParams['figure.figsize'] = (10,4)


**Pictures & intuition:**

- *Recursive* strategy feeds each prediction into the next step.
- *Direct* strategy trains one model per horizon step.

![Recursive](https://skforecast.org/0.13.0/_images/recursive_forecasting.png)

![Direct](https://skforecast.org/0.13.0/_images/direct_forecasting.png)

![Multi-output regression](https://media.geeksforgeeks.org/wp-content/uploads/20230921151937/Multioutput-Regression-in-Machine-Learning.png)


In [ ]:
def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred, squared=False)

def select_datetime_and_target(df):
    dt_candidates = [c for c in df.columns if any(k in c.lower() for k in ['time','date','datetime','timestamp'])]
    dt_col = dt_candidates[0] if dt_candidates else df.columns[0]
    df[dt_col] = pd.to_datetime(df[dt_col], errors='coerce')
    df = df.dropna(subset=[dt_col]).sort_values(dt_col).set_index(dt_col)
    num_cols = df.select_dtypes(include='number').columns.tolist()
    prefs = [c for c in num_cols if any(k in c.lower() for k in ['consump','demand','load','power','mw','kwh'])]
    y_col = prefs[0] if prefs else (num_cols[0] if num_cols else None)
    if y_col is None:
        raise ValueError('No numeric target column found in dataset.')
    return df[[y_col]].rename(columns={y_col:'y'})

def seasonal_naive(y, season=24):
    return y.shift(season)


# Get data

In [ ]:
raw = pd.read_csv(r'data/electricity-production and consumption_2022.csv')
df = select_datetime_and_target(raw)
df = df.asfreq('H')
df['y'] = df['y'].interpolate(limit_direction='both')
df.head()

# Distribution

In [ ]:
ax = df['y'].plot(kind='hist', bins=40, alpha=0.6)
ax.set_title('Demand distribution (histogram)')
plt.show()
df['y'].plot(kind='line', title='Demand (sample)')
plt.show()

# Autocorrelation

In [ ]:
plot_acf(df['y'].dropna(), lags=200)
plt.title('ACF up to 200 lags')
plt.show()
print('Strong daily/weekly spikes suggest seasonal lags like 24 and 168.')

# Baseline. Seasonal Naive Forecasting

In [ ]:
test_horizon_days = 14
split_point = df.index.max() - pd.Timedelta(days=test_horizon_days)
train, test = df.loc[:split_point].copy(), df.loc[split_point+pd.Timedelta(hours=1):].copy()

y_hat_snaive = seasonal_naive(test['y'], season=24)
if y_hat_snaive.isna().any():
    init = train['y'].iloc[-24:]
    y_hat_snaive.iloc[:24] = init.values

print('SNaive MAE:', round(mean_absolute_error(test['y'], y_hat_snaive), 3))
print('SNaive RMSE:', round(rmse(test['y'], y_hat_snaive), 3))

test[['y']].assign(SNaive=y_hat_snaive).iloc[:168].plot(title='First week of test: SNaive vs actual')
plt.show()

# One-step-ahead

In [ ]:
lags = 48
reg = XGBRegressor(n_estimators=600, max_depth=6, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9, random_state=42)
fa = ForecasterAutoreg(regressor=reg, lags=lags)
fa.fit(y=train['y'])
pred_osa = fa.predict(steps=len(test))
pred_osa.index = test.index
print('One-step MAE:', round(mean_absolute_error(test['y'], pred_osa), 3))
print('One-step RMSE:', round(rmse(test['y'], pred_osa), 3))
test[['y']].assign(OneStep=pred_osa).iloc[:168].plot(title='First week: One-step vs actual')
plt.show()

# Multi-step-ahead forecast. Direct

In [ ]:
steps = 24
fd = ForecasterAutoregDirect(regressor=reg, lags=lags, steps=steps)
fd.fit(y=train['y'])
preds_direct = []
block_starts = pd.date_range(start=test.index.min(), end=test.index.max(), freq=f'{steps}H')
for start in block_starts:
    block_idx = pd.date_range(start=start, periods=steps, freq='H')
    if block_idx[-1] > test.index.max():
        break
    y_block = fd.predict(steps=steps)
    y_block.index = block_idx
    preds_direct.append(y_block)
    end_idx = block_idx[-1]
    fa.fit(y=df.loc[:end_idx]['y'])
pred_direct = pd.concat(preds_direct).sort_index()
align = test['y'].loc[pred_direct.index]
print('Direct MAE:', round(mean_absolute_error(align, pred_direct), 3))
print('Direct RMSE:', round(rmse(align, pred_direct), 3))
align.to_frame('y').assign(Direct=pred_direct).iloc[:168].plot(title='First week: Direct vs actual')
plt.show()

# Multi-step-ahead forecast. Recursive

In [ ]:
fr = ForecasterAutoreg(regressor=reg, lags=lags)
fr.fit(y=train['y'])
preds_rec = []
for start in block_starts:
    block_idx = pd.date_range(start=start, periods=steps, freq='H')
    if block_idx[-1] > test.index.max():
        break
    y_block = fr.predict(steps=steps)
    y_block.index = block_idx
    preds_rec.append(y_block)
    end_idx = block_idx[-1]
    fr.fit(y=df.loc[:end_idx]['y'])
pred_recursive = pd.concat(preds_rec).sort_index()
align2 = test['y'].loc[pred_recursive.index]
print('Recursive MAE:', round(mean_absolute_error(align2, pred_recursive), 3))
print('Recursive RMSE:', round(rmse(align2, pred_recursive), 3))
align2.to_frame('y').assign(Recursive=pred_recursive).iloc[:168].plot(title='First week: Recursive vs actual')
plt.show()

# Split

In [ ]:
fig, ax = plt.subplots()
train['y'].iloc[-24*7:].plot(ax=ax, label='Train (last 7 days)')
test['y'].plot(ax=ax, label='Test', alpha=0.8)
ax.axvline(train.index.max(), color='k', linestyle='--', label='Split')
ax.set_title('Train/Test split overview')
ax.legend(); plt.show()

summary = pd.DataFrame({
    'Model': ['SNaive','OneStep','Direct','Recursive'],
    'MAE': [
        mean_absolute_error(test['y'], seasonal_naive(test['y'], 24).fillna(method='bfill')),
        mean_absolute_error(test['y'], pred_osa.reindex(test.index).ffill()),
        mean_absolute_error(align, pred_direct),
        mean_absolute_error(align2, pred_recursive)
    ],
    'RMSE': [
        rmse(test['y'], seasonal_naive(test['y'], 24).fillna(method='bfill')),
        rmse(test['y'], pred_osa.reindex(test.index).ffill()),
        rmse(align, pred_direct),
        rmse(align2, pred_recursive)
    ]
}).round(3)
summary